# Varejo: Detecção e Segmentação de Produtos em Prateleira
SKU-110K — Fases 1 a 4

Leonardo Ramos Coutinho | Visão Computacional e Reconhecimento de Padrões

Runtime **None** até a Fase 2 (GPU só é necessária a partir do treino). Em caso de queda de sessão ou troca de runtime, execute novamente a célula de Setup abaixo antes de continuar.


In [ ]:
# Setup
!pip install -q ultralytics pandas matplotlib tqdm pillow

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
CACHE_DIR = Path("/content/drive/MyDrive/varejo_sku110k_subset")
img_cache = CACHE_DIR / "images"
man_cache = CACHE_DIR / "manifests"
ann_cache = CACHE_DIR / "annotations_full"
img_cache.mkdir(parents=True, exist_ok=True)
man_cache.mkdir(parents=True, exist_ok=True)


## Fase 1: Dados e EDA


In [ ]:
if (man_cache / "subset_a_train.csv").exists():
    print("recorte já existe no drive, pode pular direto pra Fase 2 se já tiver treinado")
else:
    print("nada cacheado ainda, segue o notebook normal")


In [ ]:
# Carrega as anotações completas do SKU-110K: do cache do Drive se existir, senão baixa e extrai o dataset
import pandas as pd

cols = ["image_name", "x1", "y1", "x2", "y2", "class", "image_width", "image_height"]
DATA_DIR = Path("/content/datasets")
extract_dir = DATA_DIR / "SKU110K_fixed"

if (ann_cache / "annotations_train.csv").exists():
    df_train = pd.read_csv(ann_cache / "annotations_train.csv")
    df_val   = pd.read_csv(ann_cache / "annotations_val.csv")
    df_test  = pd.read_csv(ann_cache / "annotations_test.csv")
else:
    import tarfile, time
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    url = "http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz"
    tar_path = DATA_DIR / "SKU110K_fixed.tar.gz"

    if not tar_path.exists():
        t0 = time.time()
        !wget -q --show-progress -O {tar_path} {url}
        print(f"download concluído em {time.time()-t0:.0f}s")

    if not extract_dir.exists():
        with tarfile.open(tar_path) as tar:
            tar.extractall(DATA_DIR)

    ann_dir = extract_dir / "annotations"
    df_train = pd.read_csv(ann_dir / "annotations_train.csv", names=cols)
    df_val   = pd.read_csv(ann_dir / "annotations_val.csv",   names=cols)
    df_test  = pd.read_csv(ann_dir / "annotations_test.csv",  names=cols)

    df_train.to_csv(ann_cache / "annotations_train.csv", index=False)
    df_val.to_csv(ann_cache / "annotations_val.csv", index=False)
    df_test.to_csv(ann_cache / "annotations_test.csv", index=False)

print("caixas por split -> train:", len(df_train), "val:", len(df_val), "test:", len(df_test))
df_train.head()


In [ ]:
# Densidade (objetos por imagem) por split de treino
density_train = df_train.groupby("image_name").size().rename("n_objects")
print(density_train.describe())


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(density_train, bins=40)
ax[0].set_title("objetos por imagem (train)")

ax[1].hist(density_train, bins=40)
ax[1].set_xlim(0, 60)
ax[1].set_title("zoom: 0-60 objetos")
plt.tight_layout()

fig_dir = CACHE_DIR / "figuras"
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "densidade_histograma.png", dpi=150, bbox_inches='tight')
plt.show()

for cutoff in [30, 40, 50, 60]:
    print(f"imagens com <= {cutoff} objetos:", (density_train <= cutoff).sum())


In [ ]:
# Verificação de integridade das anotações
bad_dims = df_train[(df_train.x2 <= df_train.x1) | (df_train.y2 <= df_train.y1)]
out_of_bounds = df_train[(df_train.x2 > df_train.image_width) | (df_train.y2 > df_train.image_height)]

print("caixas inválidas:", len(bad_dims))
print("caixas fora dos limites:", len(out_of_bounds))
print("resoluções distintas:", df_train[['image_width','image_height']].drop_duplicates().shape[0])


In [ ]:
# Subconjunto A (detecção): amostra aleatória de cada split oficial, seed fixa (evita vazamento entre splits)
# Subconjunto B (segmentação): imagens com 20-X objetos originais (piso evita instâncias insuficientes
# para treino; teto sobe automaticamente até reunir 28 imagens). train_7082.jpg é fixada por já ter
# sido anotada integralmente (ver nota metodológica).
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

N_A_TRAIN, N_A_VAL, N_A_TEST = 350, 75, 75
N_B = 28
MIN_OBJECTS_B = 20
max_cutoff = 50

def sample_images(df, n, rng):
    imgs = df["image_name"].unique()
    n = min(n, len(imgs))
    return pd.Series(rng.choice(imgs, size=n, replace=False))

subset_a_train = sample_images(df_train, N_A_TRAIN, rng)
subset_a_val   = sample_images(df_val,   N_A_VAL,   rng)
subset_a_test  = sample_images(df_test,  N_A_TEST,  rng)

candidates = density_train[(density_train >= MIN_OBJECTS_B) & (density_train <= max_cutoff)]
while len(candidates) < N_B and max_cutoff < 200:
    max_cutoff += 5
    candidates = density_train[(density_train >= MIN_OBJECTS_B) & (density_train <= max_cutoff)]

JA_ANOTADAS = ["train_7082.jpg"]
pool_restante = candidates.drop(index=[n for n in JA_ANOTADAS if n in candidates.index])
sorteadas = rng.choice(pool_restante.index, size=min(N_B - len(JA_ANOTADAS), len(pool_restante)), replace=False)
subset_b = density_train.loc[list(JA_ANOTADAS) + list(sorteadas)].sort_values()

print("A -> train:", len(subset_a_train), "val:", len(subset_a_val), "test:", len(subset_a_test))
print("B -> faixa [", MIN_OBJECTS_B, ",", max_cutoff, "] ->", len(subset_b), "imagens")
print("B -> densidade média:", subset_b.mean().round(1), "| mínima:", subset_b.min(), "| máxima:", subset_b.max())


In [ ]:
subset_a_train.to_csv(man_cache / "subset_a_train.csv", index=False, header=["image_name"])
subset_a_val.to_csv(man_cache / "subset_a_val.csv", index=False, header=["image_name"])
subset_a_test.to_csv(man_cache / "subset_a_test.csv", index=False, header=["image_name"])
subset_b.reset_index().rename(columns={"index": "image_name"}).to_csv(man_cache / "subset_b.csv", index=False)


In [ ]:
# Copia para o Drive apenas as imagens necessárias (A + B); baixa e extrai o dataset completo só se faltar algo
import shutil
from tqdm import tqdm

all_needed = set(subset_a_train) | set(subset_a_val) | set(subset_a_test) | set(subset_b.index)
faltando = [n for n in all_needed if not (img_cache / n).exists()]

if faltando and not extract_dir.exists():
    import tarfile, time
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    url = "http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz"
    tar_path = DATA_DIR / "SKU110K_fixed.tar.gz"
    if not tar_path.exists():
        t0 = time.time()
        !wget -q --show-progress -O {tar_path} {url}
        print(f"download concluído em {time.time()-t0:.0f}s")
    with tarfile.open(tar_path) as tar:
        tar.extractall(DATA_DIR)

if faltando:
    for name in tqdm(faltando, desc="copiando pro drive"):
        src = extract_dir / "images" / name
        dst = img_cache / name
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

df_all = pd.concat([df_train, df_val, df_test])
df_subset = df_all[df_all.image_name.isin(all_needed)]
df_subset.to_csv(man_cache / "annotations_subset_a.csv", index=False)

print(len(all_needed), "imagens no cache,", len(df_subset), "linhas de anotação (vs", len(df_all), "no dataset completo)")


In [ ]:
# Opcional: libera espaço apagando o dataset completo do disco local (o cache no Drive é independente)
import shutil as _shutil

# _shutil.rmtree(extract_dir, ignore_errors=True)
# (DATA_DIR / "SKU110K_fixed.tar.gz").unlink(missing_ok=True)


## Fase 2: Detecção

Requer GPU. Troque o runtime e execute a célula de Setup novamente antes de continuar.


In [ ]:
# Reconstrói pastas de imagem/label do Subconjunto A a partir do cache no Drive
import pandas as pd

subset_a_train = pd.read_csv(man_cache / "subset_a_train.csv")["image_name"]
subset_a_val   = pd.read_csv(man_cache / "subset_a_val.csv")["image_name"]
subset_a_test  = pd.read_csv(man_cache / "subset_a_test.csv")["image_name"]
df_subset = pd.read_csv(man_cache / "annotations_subset_a.csv")

DATASET_ROOT = Path("/content/dataset_yolo")
splits = {"train": subset_a_train, "val": subset_a_val, "test": subset_a_test}

import shutil
for split, names in splits.items():
    (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)
    for name in names:
        src = img_cache / name
        dst = DATASET_ROOT / "images" / split / name
        if src.exists() and not dst.exists():
            shutil.copy2(src, dst)

print(len(subset_a_train), len(subset_a_val), len(subset_a_test), "imagens, ", len(df_subset), "caixas")


In [ ]:
# Converte anotações (x1,y1,x2,y2 absoluto) para formato YOLO (classe única) e gera data.yaml
def to_yolo_line(row):
    xc = (row.x1 + row.x2) / 2 / row.image_width
    yc = (row.y1 + row.y2) / 2 / row.image_height
    w  = (row.x2 - row.x1) / row.image_width
    h  = (row.y2 - row.y1) / row.image_height
    return f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"

for split, names in splits.items():
    names_set = set(names)
    df_split = df_subset[df_subset.image_name.isin(names_set)]
    for name, group in df_split.groupby("image_name"):
        label_path = DATASET_ROOT / "labels" / split / (Path(name).stem + ".txt")
        lines = [to_yolo_line(r) for r in group.itertuples()]
        label_path.write_text("\n".join(lines))

data_yaml = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['produto']
"""
(DATASET_ROOT / "data.yaml").write_text(data_yaml)
print(data_yaml)


Teste rápido (1 época) antes do treino completo.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
smoke = model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=8,
    seed=42,
    project="/content/runs_varejo",
    name="smoke_test",
)


Treino completo. `project` aponta para o Drive; checkpoints ficam salvos a cada época.

Em caso de queda no meio do treino, retome com:
```python
model = YOLO(f"{DRIVE_RUNS}/deteccao_baseline/weights/last.pt")
results = model.train(resume=True)
```


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
DRIVE_RUNS = "/content/drive/MyDrive/varejo_sku110k_subset/runs"

results = model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=20,
    seed=42,
    project=DRIVE_RUNS,
    name="deteccao_baseline",
)


In [ ]:
metrics = model.val(project=DRIVE_RUNS, name="deteccao_baseline_val")
print(metrics.box.map50, metrics.box.map, metrics.box.mp, metrics.box.mr)


## Fase 3: Segmentação

Anotação manual das imagens do Subconjunto B (fora do notebook, via Roboflow).


In [ ]:
# Empacota as imagens do Subconjunto B para anotação no Roboflow
import pandas as pd
import zipfile

subset_b_names = pd.read_csv(man_cache / "subset_b.csv")["image_name"]

zip_path = Path("/content/subconjunto_b.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    for name in subset_b_names:
        zf.write(img_cache / name, arcname=name)

print(len(subset_b_names), "imagens zipadas em", zip_path)

from google.colab import files
files.download(str(zip_path))


Referência visual (caixas originais do SKU-110K) para orientar a anotação manual de cada imagem.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

df_subset = pd.read_csv(man_cache / "annotations_subset_a.csv")
subset_b_names = pd.read_csv(man_cache / "subset_b.csv")["image_name"]

def show_reference(image_name):
    img = Image.open(img_cache / image_name)
    boxes = df_subset[df_subset.image_name == image_name]
    fig, ax = plt.subplots(1, figsize=(10, 10))
    ax.imshow(img)
    for row in boxes.itertuples():
        rect = patches.Rectangle((row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
    ax.set_title(f"{image_name} - {len(boxes)} objetos originais (SKU-110K)")
    ax.axis('off')
    plt.show()

for name in subset_b_names:
    show_reference(name)


Após anotar no Roboflow: gerar Version (split ~70/15/15, sem Resize, sem augmentation), exportar em formato YOLOv8 (Instance Segmentation) e enviar o zip para `varejo_sku110k_subset/roboflow_export.zip` no Drive.


In [ ]:
# Extrai o export do Roboflow e corrige o data.yaml (path relativo do Roboflow não é portável)
import zipfile

export_zip = CACHE_DIR / "roboflow_export.zip"
SEG_ROOT = Path("/content/dataset_seg")

with zipfile.ZipFile(export_zip) as zf:
    zf.extractall(SEG_ROOT)

for split in ["train", "valid", "test"]:
    n = len(list((SEG_ROOT / split / "labels").glob("*.txt")))
    print(f"{split}: {n} labels")

data_yaml_seg = f"""path: {SEG_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['bebidas', 'outros', 'snacks_embalados']
"""
(SEG_ROOT / "data.yaml").write_text(data_yaml_seg)
print(data_yaml_seg)


Teste rápido (1 época) antes do treino completo.


In [ ]:
from ultralytics import YOLO

model_seg = YOLO("yolo26n-seg.pt")
smoke_seg = model_seg.train(
    data=str(SEG_ROOT / "data.yaml"),
    epochs=1,
    imgsz=640,
    batch=8,
    seed=42,
    project="/content/runs_varejo_seg",
    name="smoke_test_seg",
)


In [ ]:
from ultralytics import YOLO

DRIVE_RUNS_SEG = "/content/drive/MyDrive/varejo_sku110k_subset/runs"

model_seg = YOLO("yolo26n-seg.pt")
results_seg = model_seg.train(
    data=str(SEG_ROOT / "data.yaml"),
    epochs=300,
    imgsz=960,
    batch=4,
    freeze=10,
    patience=50,
    seed=42,
    project=DRIVE_RUNS_SEG,
    name="segmentacao_baseline",
)


In [ ]:
metrics_seg = model_seg.val(project=DRIVE_RUNS_SEG, name="segmentacao_baseline_val")
print(metrics_seg.seg.map50, metrics_seg.seg.map, metrics_seg.seg.mp, metrics_seg.seg.mr)


In [ ]:
# Comparação visual: caixa original (SKU-110K) x máscara anotada (Roboflow)
import numpy as np

CLASS_COLORS = ['dodgerblue', 'orange', 'magenta']
fig_dir = CACHE_DIR / "figuras"
fig_dir.mkdir(parents=True, exist_ok=True)

def find_seg_files(stem):
    for split in ["train", "valid", "test"]:
        for p in (SEG_ROOT / split / "images").glob(f"{stem}*"):
            label_path = SEG_ROOT / split / "labels" / (p.stem + ".txt")
            if label_path.exists():
                return p, label_path
    return None, None

def compare_box_mask(image_name):
    stem = Path(image_name).stem
    img_path, label_path = find_seg_files(stem)
    if img_path is None:
        print(f"{image_name}: não encontrada no export do Roboflow")
        return

    img = Image.open(img_path)
    w, h = img.size
    boxes = df_subset[df_subset.image_name == image_name]
    lines = [l for l in label_path.read_text().strip().split("\n") if l]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(img)
    for row in boxes.itertuples():
        rect = patches.Rectangle((row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
    axes[0].set_title(f"{image_name} - caixa original SKU-110K ({len(boxes)} objetos)")
    axes[0].axis('off')

    axes[1].imshow(img)
    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        pts = np.array(parts[1:], dtype=float).reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        poly = patches.Polygon(pts, closed=True, edgecolor=CLASS_COLORS[cls],
                                facecolor=CLASS_COLORS[cls], alpha=0.3, linewidth=2)
        axes[1].add_patch(poly)
    axes[1].set_title(f"{image_name} - máscaras anotadas ({len(lines)} instâncias)")
    axes[1].axis('off')

    plt.tight_layout()
    out_path = fig_dir / f"{stem}_comparacao.png"
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

compare_box_mask("train_7082.jpg")
compare_box_mask("train_7716.jpg")


## Fase 4: Avaliação e Vídeo

Avaliação no conjunto de teste (nunca visto em treino ou validação).


In [ ]:
from ultralytics import YOLO

DRIVE_RUNS = "/content/drive/MyDrive/varejo_sku110k_subset/runs"
model = YOLO(f"{DRIVE_RUNS}/deteccao_baseline/weights/best.pt")
model_seg = YOLO(f"{DRIVE_RUNS}/segmentacao_baseline/weights/best.pt")


In [ ]:
metrics_test_det = model.val(data=str(DATASET_ROOT / "data.yaml"), split="test",
                              project=DRIVE_RUNS, name="deteccao_teste")
print("Detecção (teste) -> mAP50:", metrics_test_det.box.map50,
      "| mAP50-95:", metrics_test_det.box.map,
      "| precisão:", metrics_test_det.box.mp,
      "| recall:", metrics_test_det.box.mr)

metrics_test_seg = model_seg.val(data=str(SEG_ROOT / "data.yaml"), split="test",
                                  project=DRIVE_RUNS, name="segmentacao_teste")
print("Segmentação (teste) -> mAP50:", metrics_test_seg.seg.map50,
      "| mAP50-95:", metrics_test_seg.seg.map,
      "| precisão:", metrics_test_seg.seg.mp,
      "| recall:", metrics_test_seg.seg.mr)


Matrizes de confusão geradas automaticamente em `deteccao_teste/confusion_matrix.png` e `segmentacao_teste/confusion_matrix.png`.


In [ ]:
# Exemplos de predição (ground truth x modelo), salvos em figuras/
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

CLASS_COLORS = ['dodgerblue', 'orange', 'magenta']
fig_dir = CACHE_DIR / "figuras"
fig_dir.mkdir(parents=True, exist_ok=True)

rng_vis = np.random.default_rng(42)
test_names_det = list(rng_vis.choice(subset_a_test, size=4, replace=False))

for name in test_names_det:
    img_path = DATASET_ROOT / "images" / "test" / name
    boxes = df_subset[df_subset.image_name == name]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    img = Image.open(img_path)
    axes[0].imshow(img)
    for row in boxes.itertuples():
        rect = patches.Rectangle((row.x1, row.y1), row.x2 - row.x1, row.y2 - row.y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
    axes[0].set_title(f"{name} - ground truth ({len(boxes)} objetos)")
    axes[0].axis('off')

    result = model.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    axes[1].imshow(result.plot()[..., ::-1])
    axes[1].set_title(f"{name} - predição do modelo")
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(fig_dir / f"deteccao_pred_{Path(name).stem}.png", dpi=150, bbox_inches='tight')
    plt.show()

seg_test_dir = SEG_ROOT / "test" / "images"

for img_path in sorted(seg_test_dir.glob("*")):
    stem = img_path.stem
    label_path = SEG_ROOT / "test" / "labels" / (stem + ".txt")
    lines = [l for l in label_path.read_text().strip().split("\n") if l] if label_path.exists() else []

    img = Image.open(img_path)
    w, h = img.size

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(img)
    for line in lines:
        parts = line.split()
        cls = int(parts[0])
        pts = np.array(parts[1:], dtype=float).reshape(-1, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        poly = patches.Polygon(pts, closed=True, edgecolor=CLASS_COLORS[cls],
                                facecolor=CLASS_COLORS[cls], alpha=0.3, linewidth=2)
        axes[0].add_patch(poly)
    axes[0].set_title(f"{img_path.name} - ground truth ({len(lines)} instâncias)")
    axes[0].axis('off')

    result = model_seg.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    axes[1].imshow(result.plot()[..., ::-1])
    axes[1].set_title(f"{img_path.name} - predição do modelo")
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(fig_dir / f"segmentacao_pred_{stem}.png", dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# Inferência em vídeo real (executar quando os vídeos estiverem no Drive)
from ultralytics import YOLO

DRIVE_RUNS = "/content/drive/MyDrive/varejo_sku110k_subset/runs"
VIDEO_DIR = Path("/content/drive/MyDrive/varejo_sku110k_subset/videos")

model = YOLO(f"{DRIVE_RUNS}/deteccao_baseline/weights/best.pt")
model_seg = YOLO(f"{DRIVE_RUNS}/segmentacao_baseline/weights/best.pt")

videos = ["bebidas.mp4", "cosmeticos_higiene.mp4"]

for video_name in videos:
    video_path = VIDEO_DIR / video_name
    if not video_path.exists():
        print(f"{video_name} ainda não está no Drive, pulando")
        continue

    model.predict(source=str(video_path), save=True,
                   project=DRIVE_RUNS, name=f"video_deteccao_{video_path.stem}")

    model_seg.predict(source=str(video_path), save=True,
                       project=DRIVE_RUNS, name=f"video_segmentacao_{video_path.stem}")

print("inferência em vídeo concluída, resultado salvo no Drive")
